<a href="https://colab.research.google.com/github/Ashwini9713/Digital-forensics-/blob/model-lab/exp2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Network Scan Detection

This program analyzes network connection records to identify potential horizontal and vertical scanning activities.

**Horizontal Scanning**: A single source IP addresses attempts to connect to the same port on a large number of distinct destination IP addresses.

**Vertical Scanning**: A single source IP address attempts to connect to a large number of distinct ports on a single destination IP address.

In [1]:
import pandas as pd
import random
from collections import defaultdict

# --- 1. Generate Sample Network Connection Data ---
# For demonstration, let's create a synthetic dataset of network connection records.
# In a real scenario, this data would come from log files, network monitors, etc.

def generate_connection_records(num_records=1000):
    records = []
    source_ips = [f'192.168.1.{i}' for i in range(1, 10)]
    dest_ips = [f'10.0.0.{i}' for i in range(1, 50)]
    ports = [21, 22, 23, 25, 80, 139, 443, 3389, 8080]

    # Simulate some normal traffic
    for _ in range(num_records // 2):
        records.append({
            'source_ip': random.choice(source_ips),
            'destination_ip': random.choice(dest_ips),
            'destination_port': random.choice(ports)
        })

    # Simulate horizontal scan (e.g., 192.168.1.5 scanning port 22 on many hosts)
    scanner_h_ip = '192.168.1.5'
    scan_port_h = 22
    for _ in range(50):
        records.append({
            'source_ip': scanner_h_ip,
            'destination_ip': random.choice(dest_ips),
            'destination_port': scan_port_h
        })

    # Simulate vertical scan (e.g., 192.168.1.7 scanning 10.0.0.10 on many ports)
    scanner_v_ip = '192.168.1.7'
    target_v_ip = '10.0.0.10'
    for _ in range(30):
        records.append({
            'source_ip': scanner_v_ip,
            'destination_ip': target_v_ip,
            'destination_port': random.choice(ports)
        })

    return pd.DataFrame(records)

df = generate_connection_records()
display(df.head())
print(f"Generated {len(df)} connection records.")


,source_ip,destination_ip,destination_port
0,192.168.1.9,10.0.0.10,25
1,192.168.1.2,10.0.0.44,80
2,192.168.1.4,10.0.0.13,21
3,192.168.1.5,10.0.0.24,22
4,192.168.1.8,10.0.0.27,25


Generated 580 connection records.


In [2]:
# --- 2. Implement Scan Detection Logic ---

def detect_horizontal_scans(df, threshold_hosts=10, threshold_connections_per_host=1):
    horizontal_scans = {}
    # Group by source IP and destination port
    grouped = df.groupby(['source_ip', 'destination_port'])

    for (src_ip, dest_port), group in grouped:
        # Count unique destination IPs contacted by this source on this port
        unique_dest_ips = group['destination_ip'].nunique()

        # Filter out connections where source and destination are the same (self-connections)
        group_filtered = group[group['source_ip'] != group['destination_ip']]

        # Check if the number of unique destination IPs exceeds the threshold
        if unique_dest_ips >= threshold_hosts:
            # Additionally, let's ensure there are at least 'threshold_connections_per_host' attempts to each host to avoid false positives from single, legitimate connections.
            # This can be refined based on specific requirements.
            potential_scan_records = group_filtered.groupby(['destination_ip']).filter(lambda x: len(x) >= threshold_connections_per_host)
            if not potential_scan_records.empty and potential_scan_records['destination_ip'].nunique() >= threshold_hosts:
                horizontal_scans[(src_ip, dest_port)] = potential_scan_records

    return horizontal_scans

def detect_vertical_scans(df, threshold_ports=5, threshold_connections_per_port=1):
    vertical_scans = {}
    # Group by source IP and destination IP
    grouped = df.groupby(['source_ip', 'destination_ip'])

    for (src_ip, dest_ip), group in grouped:
        # Count unique destination ports contacted by this source to this destination
        unique_dest_ports = group['destination_port'].nunique()

        # Filter out connections where source and destination are the same
        group_filtered = group[group['source_ip'] != group['destination_ip']]

        # Check if the number of unique destination ports exceeds the threshold
        if unique_dest_ports >= threshold_ports:
            # Similar to horizontal, ensure multiple attempts or diverse connections if needed
            potential_scan_records = group_filtered.groupby(['destination_port']).filter(lambda x: len(x) >= threshold_connections_per_port)
            if not potential_scan_records.empty and potential_scan_records['destination_port'].nunique() >= threshold_ports:
                vertical_scans[(src_ip, dest_ip)] = potential_scan_records

    return vertical_scans





### Scan Analysis Results

In [3]:
# --- 3. Perform Analysis and Display Results ---

print("\n--- Detecting Horizontal Scans ---")
horizontal_scans_results = detect_horizontal_scans(df, threshold_hosts=5, threshold_connections_per_host=1) # Lower threshold for sample data
if horizontal_scans_results:
    for (src_ip, dest_port), records in horizontal_scans_results.items():
        print(f"\nHORIZONTAL SCAN DETECTED: Source IP '{src_ip}' scanned port '{dest_port}' on {records['destination_ip'].nunique()} unique hosts.")
        print("Supporting records (first 5):")
        display(records.head())
else:
    print("No significant horizontal scans detected.")

print("\n--- Detecting Vertical Scans ---")
vertical_scans_results = detect_vertical_scans(df, threshold_ports=3, threshold_connections_per_port=1) # Lower threshold for sample data
if vertical_scans_results:
    for (src_ip, dest_ip), records in vertical_scans_results.items():
        print(f"\nVERTICAL SCAN DETECTED: Source IP '{src_ip}' scanned destination IP '{dest_ip}' on {records['destination_port'].nunique()} unique ports.")
        print("Supporting records (first 5):")
        display(records.head())
else:
    print("No significant vertical scans detected.")



--- Detecting Horizontal Scans ---

HORIZONTAL SCAN DETECTED: Source IP '192.168.1.1' scanned port '21' on 5 unique hosts.
Supporting records (first 5):


,source_ip,destination_ip,destination_port
45,192.168.1.1,10.0.0.31,21
209,192.168.1.1,10.0.0.26,21
241,192.168.1.1,10.0.0.16,21
336,192.168.1.1,10.0.0.2,21
465,192.168.1.1,10.0.0.30,21



HORIZONTAL SCAN DETECTED: Source IP '192.168.1.1' scanned port '23' on 11 unique hosts.
Supporting records (first 5):


,source_ip,destination_ip,destination_port
9,192.168.1.1,10.0.0.7,23
37,192.168.1.1,10.0.0.8,23
60,192.168.1.1,10.0.0.34,23
144,192.168.1.1,10.0.0.42,23
225,192.168.1.1,10.0.0.46,23



HORIZONTAL SCAN DETECTED: Source IP '192.168.1.1' scanned port '80' on 7 unique hosts.
Supporting records (first 5):


...

,source_ip,destination_ip,destination_port
381,192.168.1.9,10.0.0.23,80
453,192.168.1.9,10.0.0.23,21
498,192.168.1.9,10.0.0.23,139



VERTICAL SCAN DETECTED: Source IP '192.168.1.9' scanned destination IP '10.0.0.26' on 3 unique ports.
Supporting records (first 5):


,source_ip,destination_ip,destination_port
318,192.168.1.9,10.0.0.26,80
355,192.168.1.9,10.0.0.26,139
469,192.168.1.9,10.0.0.26,25



VERTICAL SCAN DETECTED: Source IP '192.168.1.9' scanned destination IP '10.0.0.32' on 5 unique ports.
Supporting records (first 5):


,source_ip,destination_ip,destination_port
164,192.168.1.9,10.0.0.32,443
317,192.168.1.9,10.0.0.32,443
374,192.168.1.9,10.0.0.32,139
396,192.168.1.9,10.0.0.32,8080
432,192.168.1.9,10.0.0.32,21
